# Model 8 · Match evidence across sightings
Emad, 27 November 2026. Dewang's 3e supplies the candidate probabilities, using the plain stack and final blend selected in the preceding 24-episode experiment. This notebook builds additional same-animal matches from the supplied MIEW-ID and SIFT features. A query can inherit an identity from another sighting even when its own top 15 missed that identity.

The development and evaluation galleries contain disjoint animals. A separate final fit uses all labelled animals for the competition CSV. Execute the cells in order from the project directory with the Emad Python environment.

On Kaggle attach the competition metadata, MIEW-ID features and both SIFT archives. The CSV is written to /kaggle/working. Locally the notebook uses the repository inputs and writes to Models/Emad. The first run builds the SIFT cache.

Save Version → Save & Run All on Kaggle to publish the fitted model8_artifacts output. Attach that output and the competition data to model8_inference.ipynb. Local exports go under model3e_plus_cache.

In [ ]:
runtime_parts=[]
runtime_parts.append(r'''# Runtime configuration. Set ROOT explicitly if this notebook is copied elsewhere.
from pathlib import Path
import os, time, json, types, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Processed_Data/master_dataset.csv").exists()), None)
RAW = None
if ROOT is None:
    # Attach the ProjectWild3 inputs to a Kaggle notebook.
    options = [p.parent for p in Path("/kaggle/input").rglob("miewid_ids.csv")
               if (p.parent / "train.csv").is_file()]
    if len(options) != 1:
        raise FileNotFoundError("Expected one ProjectWild3 dataset under /kaggle/input or the local Processed_Data folder")
    RAW = options[0]
    ROOT = NOTEBOOK_DIR = Path("/kaggle/working")
else:
    NOTEBOOK_DIR = ROOT / "Models/Emad"
PROC = ROOT / "Processed_Data"
CACHE = NOTEBOOK_DIR / "model3e_plus_cache"
CACHE.mkdir(parents=True, exist_ok=True)
N_ROUNDS, N_SEEDS, N_TREES = 3, 3, 400
N_WORKERS, K_CANDIDATES, N_NEIGHBOURS, RANDOM_SEED = 12, 15, 150, 42
from threadpoolctl import threadpool_limits
import warnings
warnings.filterwarnings("ignore", message="Only one sample available.*")
np.random.seed(RANDOM_SEED)

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else CACHE
import joblib, hashlib, shutil
''')
exec(runtime_parts[-1])


In [ ]:
runtime_parts.append(r'''# Full Model 4c core source, embedded from the repository.
import types
c4 = types.ModuleType("model4c_inline")
_MODEL4C_SOURCE = '#!/usr/bin/env python3\n"""\nMODEL 4c (Dewang): Model 4b + variance reduction + extra evidence features.\n\nEverything in Model 4b (0.85373 public) is kept unchanged.  4c adds, in order of\nexpected value / risk:\n\n  1. Seed-bagged LightGBM  (--seeds, default 3).  Each of the three classifiers is\n     trained with several random seeds (different row/column subsamples) and the\n     probabilities are averaged.  Pure variance reduction: cannot add bias.\n  2. More simulated episodes (default 24 vs 16).  The classifiers only ever see\n     simulated open-set episodes, so more/more varied episodes = less overfit to\n     any one simulated gallery composition.\n  3. Softmax / kernel-density identity posterior.  For each query, a temperature-\n     scaled softmax over ALL gallery similarities, summed per identity (a KDE-style\n     class posterior).  Uses every image of an identity rather than just the top 1-2.\n     Also its entropy / max as query-level "how open-set is this query" evidence.\n  4. A second, diverse similarity signal: PCA-whitened embedding (Model 4\'s\n     representation, top 128 components) per-identity max similarity and rank.\n  5. Mild extra LightGBM L2 regularisation (reg_lambda=2) and min_child_samples=40.\n  6. Diagnostics (no effect on predictions):\n       - candidate recall@K on simulated episodes (is the true identity even\n         in the candidate list?  upper bound for MAP@5 on known queries)\n       - sim-vs-test covariate-shift report (which features look different on the\n         real test set than in the simulations)\n\nRules compliance: only the provided MiewID / SIFT features; PCA, LDA, kernel\ndensity / softmax scoring, RANSAC, LightGBM and bagging -- all on the permitted list.\n\nAnti-overfit safeguards (carried over from 4b): identity-disjoint simulated episodes,\nnear-duplicate-aware group holdout, regularised LDA, subsampled LightGBM,\nout-of-fold stacking.\n\nThe SIFT pair cache is shared with Model 4b (same matching parameters), so a\nprevious 4b run means no SIFT recomputation.\n\nRun:  python model4c_pipeline.py                   (validation only)\n      python model4c_pipeline.py --predict-test     (also writes submission)\n      python model4c_pipeline.py --seeds 1 --sims 16   (fast, ~4b-equivalent)\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nimport zipfile as _zipfile\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\n\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\n\nraw_sift = None\nNEW = "new_individual"\nVIEWS = ["left", "right", "front", "back", "frontleft", "frontright",\n         "backleft", "backright", "up", "down"]\nFLANK = np.array([0, 1, 2, 2, 0, 1, 0, 1, 2, 2])  # left-ish, right-ish, other\n\n# Mutable globals set by load_data / sift_matrices\nX = None          # (N, D) L2-normalised MiewID features\nIDS = None        # (N,) image ids\nY = None          # (N,) identity labels (None for test)\nSP = None         # (N,) species\nVIEW = None       # (N,) viewpoint code (int)\nGOOD = None       # sparse (N, N) SIFT ratio-test matches\nINL = None        # sparse (N, N) RANSAC inliers\n\n# Hyperparams set at runtime\nK = 15                                               # candidates per query\nN_SIMS = 24                                          # simulated episodes\nFOLDS = 4                                            # stacking folds\nROUNDS = 3                                           # stacking rounds\nSEEDS = 3                                            # LightGBM bagging seeds\nFRAC = (0.3, 0.5)                                    # simulated query-holdout fraction range\nTAU = 0.05                                           # softmax temperature (posterior)\nWHITEN_COMPS = 128                                   # PCA comps for whitened signal\nNEIGHBOURS = 150                                     # SIFT neighbourhood size\nMATCH_RATIO = 0.85                                   # SIFT ratio test\nRANSAC_PX = 12.0                                     # RANSAC threshold\nSIFT_PULL = 0.05                                     # SIFT weight in candidate selection\nREGIMES = [None, (0.95, 100), (0.99, 80), (0.9, 60)] # near-duplicate linking\nPARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,\n              min_child_samples=40, reg_lambda=2.0, subsample=0.8, subsample_freq=1,\n              colsample_bytree=0.8, n_estimators=400, verbose=-1)\nINDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\\d$")\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Data loading\n# ═══════════════════════════════════════════════════════════════════════════════\ndef unit(a):\n    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)\n\n\ndef load_data(root):\n    """Load master_dataset.csv and miewid_features.npy, populate globals."""\n    global X, IDS, Y, SP, VIEW\n    proc = os.path.join(root, "Processed_Data")\n    master = pd.read_csv(os.path.join(proc, "master_dataset.csv"))\n    raw = np.load(os.path.join(proc, "miewid_features.npy"))\n    need = {"id", "identity", "species", "viewpoint", "split"}\n    missing_cols = need - set(master.columns)\n    if missing_cols:\n        raise SystemExit(f"master_dataset.csv is missing columns {sorted(missing_cols)}; "\n                         f"found {list(master.columns)}")\n    # Align features with master_dataset order via miewid_index (fallback: row order)\n    if "miewid_index" in master.columns:\n        feat_idx = master["miewid_index"].to_numpy()\n    else:\n        print("  WARNING: no \'miewid_index\' column; assuming features are in row order")\n        feat_idx = np.arange(len(master))\n    X = unit(np.asarray(raw[feat_idx], dtype=np.float32))\n    IDS = master["id"].to_numpy()\n    Y = master["identity"].to_numpy(dtype=object)\n    SP = master["species"].to_numpy(dtype=object)\n    VIEW = np.array([{v: i for i, v in enumerate(VIEWS)}.get(str(v), -1)\n                      for v in master["viewpoint"]], dtype=int)\n    print(f"  species values: {sorted(set(map(str, SP)))}   splits: "\n          f"{master[\'split\'].value_counts().to_dict()}")\n    train = master[master["split"] == "train"].reset_index(drop=True)\n    test = master[master["split"] == "test"].reset_index(drop=True)\n    # Map to global indices in X\n    id_to_idx = {iid: i for i, iid in enumerate(IDS)}\n    train_idx = np.array([id_to_idx[i] for i in train["id"]])\n    test_idx = np.array([id_to_idx[i] for i in test["id"]])\n    return train, test, master, train_idx, test_idx\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# SIFT verification  (ThreadPoolExecutor + lru_cache — no fork, no zip issues)\n# ═══════════════════════════════════════════════════════════════════════════════\nclass SiftCache:\n    """Thread-safe SIFT descriptor loader with LRU cache.\n\n    OpenCV BFMatcher / estimateAffinePartial2D release the GIL, so\n    ThreadPoolExecutor gives real parallelism without fork-related NPZ\n    issues that multiprocessing.Pool triggers on Python 3.10+.\n    """\n\n    def __init__(self, npz_path, cache_path, workers=2, mem_items=2000):\n        from pathlib import Path\n        extracted = Path(npz_path).with_suffix("")\n        self.extracted = extracted if extracted.is_dir() else None\n        self.npz = raw_sift if raw_sift is not None else (None if self.extracted else np.load(npz_path))\n        self.npz_keys = ({p.stem for p in self.extracted.glob("*.npy")} if self.extracted else set(self.npz.files))\n        hits = sum(f"{i}_desc" in self.npz_keys for i in IDS)\n        print(f"  SIFT npz: {len(self.npz_keys)} arrays, descriptors found for "\n              f"{hits}/{len(IDS)} images", flush=True)\n        if hits == 0:\n            raise SystemExit(f"No \'<id>_desc\' keys in {npz_path}. Example keys: "\n                             f"{list(self.npz_keys)[:6]}")\n        self.read_lock = Lock()          # NpzFile\'s internal ZipFile is not thread-safe\n        self.workers = workers\n        self.cache_path = cache_path\n        self.done = {}\n        if os.path.exists(cache_path):\n            with np.load(cache_path) as z:\n                for p, v in zip(z["pairs"], z["values"]):\n                    self.done[tuple(p)] = tuple(v)\n        # Bind lru_cache *per instance* so it can be garbage-collected\n        self._descriptors = lru_cache(maxsize=mem_items)(self._descriptors_raw)\n\n    def _descriptors_raw(self, i):\n        """Load RootSIFT descriptors and keypoint positions for one image."""\n        iid = IDS[i]\n        dkey, xykey = f"{iid}_desc", f"{iid}_xy"\n        if dkey not in self.npz_keys:\n            return np.zeros((0, 128), np.float32), np.zeros((0, 2), np.float32)\n        with self.read_lock:\n            d = np.asarray((np.load(self.extracted / (dkey + ".npy")) if self.extracted else self.npz[dkey]), dtype=np.float32)\n            xy = np.asarray((np.load(self.extracted / (xykey + ".npy")) if self.extracted else self.npz[xykey]), dtype=np.float32)[:, :2]\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1e-12))\n        return np.ascontiguousarray(d), xy\n\n    def verify(self, pair):\n        """Mutual ratio-test matches and RANSAC inliers between two images."""\n        a, b = pair\n        da, xa = self._descriptors(a)\n        db, xb = self._descriptors(b)\n        if len(da) < 4 or len(db) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(db, da)])\n        good = [m[0] for m in bf.knnMatch(da, db, k=2)\n                if len(m) == 2 and m[0].distance < MATCH_RATIO * m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        src = np.float32([xa[m.queryIdx] for m in good])\n        dst = np.float32([xb[m.trainIdx] for m in good])\n        # Deterministic RANSAC per pair\n        cv2.setRNGSeed(int((int(a) * 100003 + int(b)) % (2**31 - 1)))\n        _, inliers = cv2.estimateAffinePartial2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_PX)\n        return len(good), 0 if inliers is None else int(inliers.sum())\n\n    def save(self):\n        os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)\n        pairs = sorted(self.done)\n        np.savez_compressed(\n            self.cache_path,\n            pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n\n\ndef sift_matrices(npz_path, cache_path, workers=2):\n    """Build symmetric sparse matrices of SIFT matches/inliers for all needed pairs."""\n    global GOOD, INL\n    cache = SiftCache(npz_path, cache_path, workers)\n    n = len(X)\n    # Find pairs to verify: nearest same-species neighbours\n    wanted = set()\n    for sp in ("hyena", "leopard"):\n        ix = np.where(SP == sp)[0]\n        # Chunk the similarity computation to limit memory\n        for start in range(0, len(ix), 256):\n            q = ix[start:start + 256]\n            sim = X[q] @ X[ix].T\n            # Zero out self-similarities\n            for local_i, global_i in enumerate(q):\n                col = np.searchsorted(ix, global_i)\n                if col < sim.shape[1] and ix[col] == global_i:\n                    sim[local_i, col] = -9.0\n            k = min(NEIGHBOURS, len(ix) - 1)\n            nb = np.argsort(-sim, 1)[:, :k]\n            for row, nbs in zip(q, nb):\n                wanted.update((min(int(row), int(j)), max(int(row), int(j)))\n                              for j in ix[nbs])\n    missing = sorted(wanted - cache.done.keys())\n    print(f"  SIFT: {len(wanted)} pairs needed, {len(missing)} uncached", flush=True)\n    if missing:\n        cv2.setNumThreads(1)   # each thread does its own matching\n        t0 = time.time()\n        BLOCK = 20000\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            for s0 in range(0, len(missing), BLOCK):\n                block = missing[s0:s0 + BLOCK]\n                for pair, value in zip(block, pool.map(cache.verify, block)):\n                    cache.done[pair] = value\n                cache.save()      # re-running resumes from here if interrupted\n                print(f"    verified {s0 + len(block)}/{len(missing)}  "\n                      f"({time.time() - t0:.0f}s)", flush=True)\n        cache.save()\n    # Build sparse matrices from wanted pairs only\n    pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n    vv = np.array([cache.done[tuple(p)] for p in pp]).reshape(-1, 2)\n    a, b = pp.T\n    def mat(v):\n        return sparse.coo_matrix(\n            (np.r_[v, v], (np.r_[a, b], np.r_[b, a])), shape=(n, n)).tocsr()\n    GOOD, INL = mat(vv[:, 0]), mat(vv[:, 1])\n    print(f"  SIFT matrices built: {GOOD.nnz} nonzeros", flush=True)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# PCA + LDA embedding  (upgrade #1 over Model 4)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef embed(gallery_idx, query_idx, labels):\n    """PCA -> LDA projection (+ a PCA-whitened view).  PCA is unsupervised (sees\n    queries too).  LDA is supervised on gallery labels only -> identity-disjoint."""\n    all_idx = np.r_[gallery_idx, query_idx]\n    n_comp = min(384, len(all_idx) - 1, X.shape[1])\n    key=(\'embedding\',str(SP[gallery_idx[0]]))\n    if phase == \'load\':\n        pca,lda=state[key]\n        pg,pq=pca.transform(X[gallery_idx]),pca.transform(X[query_idx])\n    else:\n        pca=PCA(n_comp).fit(X[all_idx])\n        pg,pq=pca.transform(X[gallery_idx]),pca.transform(X[query_idx])\n        lda=LDA(solver=\'eigen\',shrinkage=.3).fit(pg,labels)\n        if phase == \'save\': state[key]=(pca,lda)\n    w = min(WHITEN_COMPS, n_comp)\n    scale = 1.0 / np.sqrt(pca.explained_variance_[:w] + 1e-9)\n    return (unit(lda.transform(pg)), unit(lda.transform(pq)),\n            unit(pg[:, :w] * scale), unit(pq[:, :w] * scale))\n\n\ndef posterior(sim, inv, n, tau=None):\n    """Softmax over all gallery images, summed per identity (KDE-style posterior)."""\n    tau = TAU if tau is None else tau\n    e = np.exp((sim - sim.max(1, keepdims=True)) / tau)\n    onehot = sparse.csr_matrix((np.ones(len(inv)), (np.arange(len(inv)), inv)),\n                               shape=(len(inv), n))\n    num = np.asarray(onehot.T.dot(e.T)).T\n    return num / np.maximum(num.sum(1, keepdims=True), 1e-12)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Feature extraction  (upgrade #4 — richer features)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s images."""\n    best = np.empty((sim.shape[0], n))\n    top2 = np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j] = s.max(1)\n        top2[:, j] = np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef duplicate_groups(ix, link):\n    """Group near-duplicate images of one identity."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef species_features(gi, qi, sp):\n    """Build query-level and (query, candidate) feature DataFrames for one species."""\n    Rg, Rq = X[gi], X[qi]\n    yg, vg, vq = Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq, Wg, Wq = embed(gi, qi, yg)\n\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    if n < K:\n        raise ValueError(f"only {n} gallery identities for {sp}, need >= candidates ({K})")\n\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    sim_w = Wq @ Wg.T\n    wht1, _ = per_identity(sim_w, inv, n)\n    post_l, post_r = posterior(sim_l, inv, n), posterior(sim_r, inv, n)\n    ent_l = -(post_l * np.log(np.maximum(post_l, 1e-12))).sum(1) / np.log(max(n, 2))\n    pmax_l, pmax_r = post_l.max(1), post_r.max(1)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n\n    # SIFT per-identity features\n    inl_g = INL[qi][:, gi].toarray().astype(np.float32)\n    good_g = GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    # Support: how many gallery images of each identity have ≥7 RANSAC inliers\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]\n\n    # Same-viewpoint similarity (upgrade #4)\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n\n    # Viewpoint coverage per identity\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n\n    # Transductive: nearest query neighbours by embedding and SIFT\n    qq_r, qq_l = Rq @ Rq.T, Lq @ Lq.T\n    qq_s = INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1)\n    np.fill_diagonal(qq_l, -1)\n    nb = np.argsort(-qq_l, 1)[:, :5]\n    sb = np.argsort(-qq_s, 1)[:, :5]\n    nbs = np.take_along_axis(qq_l, nb, 1)\n    sbs = np.take_along_axis(qq_s, sb, 1)\n\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n\n    # Query-level features\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2],\n        "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "pmax_l": pmax_l, "pmax_r": pmax_r, "ent_l": ent_l,\n        "w1": wht1.max(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1),\n        "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1),\n        "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"] = nb[:, k], nbs[:, k]\n        q[f"sb{k}"], q[f"sw{k}"] = sb[:, k], sbs[:, k]\n\n    # Candidate-level features (upgrade #4)\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None],\n                               np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None],\n                               np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c],\n        "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0],\n        "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c],\n        "support": support[r, c],\n        "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c],\n        "sgap": sift1[r, c] - t_s[r, 0],\n        "pl": post_l[r, c], "pr": post_r[r, c],\n        "pgap": pmax_l[r] - post_l[r, c],\n        "mxw": wht1[r, c], "rk_w": rank_of(wht1),\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    """Build feature frames across both species."""\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g = gi[SP[gi] == sp]\n        qq = qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q)\n        cs.append(c)\n        order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True),\n                c=pd.concat(cs, ignore_index=True),\n                order=np.concatenate(order))\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Simulated open-set episodes  (upgrade #5 — regime-diverse)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef simulate(seed, train_idx):\n    """Identity-disjoint open-set split with regime-mixture simulation."""\n    rng = np.random.default_rng(seed)\n    link = REGIMES[seed % len(REGIMES)]\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        p_unknown = rng.uniform(0.2, 0.35)\n        frac = rng.uniform(*FRAC)\n        cap = int(rng.integers(12, 33))\n        ix = train_idx[SP[train_idx] == sp]\n        names = np.array(sorted(set(Y[ix])))\n        rng.shuffle(names)\n        new_set = set(names[:int(round(p_unknown * len(names)))])\n        for name in names:\n            rows = ix[Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in new_set:\n                unknown.extend(rows[:cap].tolist())\n            elif len(rows) < 2:\n                gallery.extend(rows.tolist())\n            else:\n                n_q = min(max(int(round(frac * len(rows))), 1), cap, len(rows) - 1)\n                groups = duplicate_groups(rows, link)\n                held = []\n                for c in rng.permutation(np.unique(groups)):\n                    members = rows[groups == c].tolist()\n                    if len(held) < n_q and len(held) + len(members) < len(rows):\n                        held.extend(members)\n                known.extend(held[:cap])\n                held_set = set(held)\n                gallery.extend(i for i in rows if i not in held_set)\n    gi = np.array(gallery, dtype=int)\n    qi = np.array(known + unknown, dtype=int)\n    frames = build_frames(gi, qi)\n    truth = np.where(np.isin(frames["order"], unknown), NEW, Y[frames["order"]])\n    frames["truth"] = truth\n    frames["q"]["unk"] = (truth == NEW).astype(int)\n    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) ==\n                        truth[frames["c"].q.to_numpy()]).astype(int)\n    return frames\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# LightGBM stacked classifiers  (upgrade #2 — replaces logistic regression)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef columns(s):\n    # "ng" (gallery size) is excluded: it is a per-episode fingerprint in the sims and\n    # far outside the simulated range at test time (gallery = full train).\n    qf = [c for c in s["q"].columns\n          if c not in ("unk", "ng") and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True),\n                       s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\nclass Bag:\n    """Average predict_proba over several LightGBM models (different seeds)."""\n\n    def __init__(self, models):\n        self.models = models\n\n    def predict_proba(self, x):\n        return np.mean([m.predict_proba(x) for m in self.models], axis=0)\n\n\ndef bagged(params, x, y):\n    base = params.get("random_state", 0) or 0\n    return Bag([lgb.LGBMClassifier(**{**params, "random_state": base + 101 * k}).fit(x, y)\n                for k in range(SEEDS)])\n\n\ndef fit(sims):\n    """Fit new-individual detector, closed-set candidate scorer, all-query scorer\n    (each a bag of SEEDS LightGBM models)."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = bagged({**PARAMS, "n_estimators": 300, "num_leaves": 15},\n                      pd.concat([s["q"][qf] for s in sims]),\n                      np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    closed_scorer = bagged(PARAMS, cx[ck], cy[ck])\n    all_scorer = bagged(PARAMS, cx, cy)\n    return detector, closed_scorer, all_scorer, (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq = len(s["q"])\n    cx = candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within,\n                pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([\n        s["c"].ident.to_numpy(dtype=object).reshape(nq, K),\n        np.full((nq, 1), NEW, dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"],\n                              r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef candidate_recall(s):\n    """Fraction of known-identity queries whose true identity is among the K candidates\n    (hard upper bound on known-query MAP@5)."""\n    nq = len(s["q"])\n    cand = s["c"].ident.to_numpy(dtype=object).reshape(nq, K)\n    known = s["truth"] != NEW\n    hit = (cand == s["truth"][:, None]).any(1)\n    return float(hit[known].mean()) if known.any() else float("nan")\n\n\ndef shift_report(sims, final, top=8):\n    """Compare query-level features of the real test frame with the simulations.\n    Big standardised shifts = the classifiers see test data unlike their training."""\n    cols = [c for c in final["q"].columns\n            if c in sims[0]["q"].columns and c not in ("unk", "ng")\n            and not INDEX_COLS.match(c)]\n    a = pd.concat([s["q"][cols] for s in sims]).astype(float)\n    b = final["q"][cols].astype(float)\n    sd = a.std().replace(0, np.nan)\n    z = ((b.median() - a.median()) / sd).dropna()\n    z = z.reindex(z.abs().sort_values(ascending=False).index)\n    print("  covariate shift (test median - sim median) / sim std, largest first:")\n    for name, v in z.head(top).items():\n        print(f"    {name:10s} {v:+.2f}   (sim {a[name].median():.3f}  test {b[name].median():.3f})")\n    if z.abs().head(top).max() > 1.0:\n        print("    -> |shift| > 1 on some features: consider widening simulate() regimes")\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0)\n                    for row, t in zip(preds, truth)])\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Transductive stacking  (upgrade #3)\n# ═══════════════════════════════════════════════════════════════════════════════\ndef augment(s, r):\n    """Add previous round\'s predictions as features for the next round."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"] = r["pn"]\n    q["r_pb"] = r["pb"]\n    q["r_rawsum"] = r["raw"].sum(1)\n    q["r_rawmax"] = r["raw"].max(1)\n    c["r_raw"] = r["raw"].ravel()\n    c["r_pc"] = r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):\n        evidence = np.zeros((nq, K))\n        support = np.zeros((nq, K))\n        pb_list, w_list, top_list = [], [], []\n        for k in range(5):\n            idx = q[f"{kind}{k}"].to_numpy()\n            sim = q[f"{weight}{k}"].to_numpy() / scale\n            wk = (np.clip(sim - 0.4, 0, None) if kind == "nb"\n                  else np.clip(sim, 0, 1))\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) *\n                     r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence,\n                                   np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb_list.append(r["pb"][idx])\n            w_list.append(wk)\n            top_list.append(r["raw"][idx].max(1))\n        pb = np.stack(pb_list, 1)\n        w = np.stack(w_list, 1)\n        q[f"{kind}_pb0"] = pb[:, 0]\n        q[f"{kind}_pbw"] = (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"] = np.where(w > 0, pb, 1).min(1)\n        q[f"{kind}_w"] = w.sum(1)\n        q[f"{kind}_rm0"] = top_list[0]\n        c[f"r_{kind}ev"] = evidence.ravel()\n        c[f"r_{kind}sum"] = support.ravel()\n    return dict(s, q=q, c=c)\n\n\n# ═══════════════════════════════════════════════════════════════════════════════\n# Validation & submission\n# ═══════════════════════════════════════════════════════════════════════════════\ndef validate_submission(sub, test_df, train_df, sample_path):\n    assert len(sub) == len(test_df), "row count mismatch"\n    assert sub["id"].is_unique, "duplicate ids"\n    assert set(sub["id"]) == set(test_df["id"]), "ids differ from test ids"\n    if sample_path and os.path.exists(sample_path):\n        sample = pd.read_csv(sample_path)\n        assert list(sample.columns) == list(sub.columns), \\\n            f"columns {list(sub.columns)} != sample {list(sample.columns)}"\n        assert (sub["id"].values == sample["id"].values).all(), \\\n            "id order differs from sample_submission"\n    else:\n        print("  (sample_submission.csv not found: only internal checks done)")\n    known = {sp: set(train_df[train_df["species"] == sp]["identity"])\n             for sp in ("hyena", "leopard")}\n    sp_of = dict(zip(test_df["id"], test_df["species"]))\n    for i, pr in zip(sub["id"], sub["predictions"]):\n        t = pr.split()\n        assert len(t) == 5 and len(set(t)) == 5, f"bad row {i}"\n        for x in t:\n            assert x == NEW or x in known[sp_of[i]], \\\n                f"wrong-species/unknown label {x} in {i}"\n    print("  submission checks passed")\n\n\n'
exec(compile(_MODEL4C_SOURCE, "<embedded Model 4c>", "exec"), c4.__dict__)

# Model 3e methods and extensions, embedded; main()/CLI block is omitted.
_MODEL3E_SOURCE = '#!/usr/bin/env python3\n"""\nMODEL 3e: Model 4c + the gaps listed in the team\'s improvement note (26 Nov).\nKeep model4c_pipeline.py in the same folder.\n\n  1. Test mismatch      simulated queries follow the test viewpoint mix; a test-vs-sim\n                        classifier measures the remaining gap and gives importance\n                        weights, used only if they help weighted validation.\n  2. Validation         folds are split by ANIMAL, not by simulated episode.\n  3. One ranker         LightGBM lambdarank over K candidates + a new_individual row;\n                        compared with 4c\'s hand-made fusion and a 50/50 blend.\n  4. Background SIFT    generic keypoints (similar to descriptors in many images) are\n                        down-weighted.\n  5. More SIFT          weighted inliers, inlier hull area, rotation/scale\n                        consistency, affine-model inliers.\n  6. PLDA               two-covariance likelihood ratio per (query, identity).\n\nNo leaderboard probing: every choice is made on training-set validation and one\nsubmission file is written.\n\nRules: provided features only; numpy / scipy / scikit-learn / LightGBM / OpenCV\nmatching.  Item 1 uses UNLABELLED test features and metadata: confirm with the TA\nteam, or pass --no-test-adapt.\n\nValidation caveat: neighbour-stacking features still come from first-level models\nthat saw other folds (ordinary stacking, not nested), so scores stay slightly\noptimistic.  They are not comparable with 4c\'s episode-split numbers.\n\nRun:  python model3e_pipeline.py                  validation only\n      python model3e_pipeline.py --predict-test   validation + submission\n      add --retune to search LightGBM size/regularisation on the animal split\n      add --quick for one seed (fast first look)\nThe first run recomputes SIFT pairs (new statistics, new cache file).\n"""\nimport argparse\nimport json\nimport os\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.linalg import eigh\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\n# Model 4c runtime is embedded in the previous cell as c4\n\nNEW = c4.NEW\nSPECIES = ("hyena", "leopard")\nEXTRA = {}                  # name -> sparse (N, N) matrix of extra SIFT statistics\nEXTRA_NAMES = ["wsift", "hull", "rig", "aff"]\nEXTRA_PREFIX = tuple(f"{n}_" for n in EXTRA_NAMES + ["plda"])\n_base_species_features = c4.species_features\n\n\n# ═══ SIFT: keypoint distinctiveness + richer pair statistics ═════════════════\nclass SiftCache3e(c4.SiftCache):\n    """4c\'s matcher (same good/inlier counts) with extra per-pair statistics."""\n\n    def __init__(self, npz_path, cache_path, workers):\n        super().__init__(npz_path, cache_path, workers, mem_items=2000)\n        self.weights = {}\n\n    def genericness(self, train_idx, path, n_ref=200, per_ref=100, top=10):\n        """For each keypoint: mean similarity to its closest descriptor in each of the\n        `top` most similar reference TRAIN images.  High = common texture."""\n        g = {}\n        if os.path.exists(path):\n            with np.load(path) as z:\n                g = {int(k): z[k] for k in z.files}\n        rng = np.random.default_rng(0)\n        for sp in SPECIES:\n            todo = [i for i in np.where(c4.SP == sp)[0] if i not in g]\n            if not todo:\n                continue\n            ref_ids, blocks = [], []\n            for i in rng.permutation(train_idx[c4.SP[train_idx] == sp]):\n                d = c4.SiftCache._descriptors_raw(self, int(i))[0]\n                if len(d):\n                    ref_ids.append(int(i))\n                    blocks.append(d[rng.choice(len(d), per_ref, replace=len(d) < per_ref)])\n                if len(ref_ids) == n_ref:\n                    break\n            ref = np.concatenate(blocks)\n            column = {i: j for j, i in enumerate(ref_ids)}\n            print(f"  keypoint weights, {sp}: {len(todo)} images vs "\n                  f"{len(ref_ids)} reference images", flush=True)\n            for n, i in enumerate(todo, 1):\n                d = c4.SiftCache._descriptors_raw(self, int(i))[0]\n                out = np.zeros(len(d), np.float32)\n                for s in range(0, len(d), 512):\n                    sim = d[s:s + 512] @ ref.T\n                    best = sim.reshape(len(sim), len(ref_ids), per_ref).max(2)\n                    if i in column:\n                        best[:, column[i]] = -1.0\n                    k = min(top, best.shape[1])\n                    out[s:s + 512] = np.partition(best, -k, 1)[:, -k:].mean(1)\n                g[int(i)] = out\n                if n % 500 == 0 or n == len(todo):\n                    print(f"    {n}/{len(todo)}", flush=True)\n                    os.makedirs(os.path.dirname(path), exist_ok=True)\n                    np.savez_compressed(path, **{str(k): v for k, v in g.items()})\n        for sp in SPECIES:\n            ix = [i for i in np.where(c4.SP == sp)[0] if len(g.get(i, ()))]\n            if not ix:\n                continue\n            lo, hi = np.quantile(np.concatenate([g[i] for i in ix]), [0.5, 0.95])\n            for i in ix:\n                self.weights[int(i)] = np.clip(\n                    (hi - g[i]) / max(hi - lo, 1e-6), 0.05, 1.0).astype(np.float32)\n\n    def _descriptors_raw(self, i):\n        d, xy = super()._descriptors_raw(i)\n        w = self.weights.get(int(i), np.ones(len(d), np.float32))\n        area = (cv2.contourArea(cv2.convexHull(np.ascontiguousarray(xy)))\n                if len(xy) >= 3 else 0.0)\n        return d, xy, w, area\n\n    def verify(self, pair):\n        """(good, inliers, weighted inliers, hull fraction, rigid score, affine inliers)"""\n        a, b = pair\n        da, xa, wa, area_a = self._descriptors(a)\n        db, xb, wb, area_b = self._descriptors(b)\n        if len(da) < 4 or len(db) < 4:\n            return 0, 0, 0.0, 0.0, 0.0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(db, da)])\n        good = [m[0] for m in bf.knnMatch(da, db, k=2)\n                if len(m) == 2 and m[0].distance < c4.MATCH_RATIO * m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0, 0.0, 0.0, 0.0, 0\n        qi = np.array([m.queryIdx for m in good])\n        ti = np.array([m.trainIdx for m in good])\n        src, dst = np.float32(xa[qi]), np.float32(xb[ti])\n        cv2.setRNGSeed(int((int(a) * 100003 + int(b)) % (2**31 - 1)))\n        M, mask = cv2.estimateAffinePartial2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=c4.RANSAC_PX)\n        if M is None or mask is None:\n            return len(good), 0, 0.0, 0.0, 0.0, 0\n        mask = mask.ravel().astype(bool)\n        inl = int(mask.sum())\n        winl = float(np.minimum(wa[qi[mask]], wb[ti[mask]]).sum())\n        hull = rigid = 0.0\n        if inl >= 5:\n            fr = [cv2.contourArea(cv2.convexHull(p[mask])) / max(area, 1e-6)\n                  for p, area in ((src, area_a), (dst, area_b))]\n            hull = float(np.clip(min(fr), 0, 1))\n            angle = abs(np.arctan2(M[1, 0], M[0, 0]))\n            logscale = abs(np.log(max(np.hypot(M[0, 0], M[1, 0]), 1e-6)))\n            rigid = float(np.exp(-angle / 0.5 - logscale / 0.5) * np.log1p(winl))\n        _, mask2 = cv2.estimateAffine2D(\n            src, dst, method=cv2.RANSAC, ransacReprojThreshold=c4.RANSAC_PX)\n        return (len(good), inl, winl, hull, rigid,\n                0 if mask2 is None else int(mask2.sum()))\n\n    def save(self):\n        os.makedirs(os.path.dirname(self.cache_path) or ".", exist_ok=True)\n        pairs = sorted(self.done)\n        np.savez_compressed(\n            self.cache_path,\n            pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n            values=np.array([self.done[p] for p in pairs], float).reshape(-1, 6))\n\n\ndef sift_matrices(npz_path, cache_dir, train_idx, workers):\n    cache = SiftCache3e(npz_path, os.path.join(cache_dir, "sift_pairs_3e.npz"), workers)\n    cv2.setNumThreads(1)\n    n = len(c4.X)\n    wanted = set()\n    for sp in SPECIES:                       # same neighbourhood rule as 4c\n        ix = np.where(c4.SP == sp)[0]\n        k = min(c4.NEIGHBOURS, len(ix) - 1)\n        for start in range(0, len(ix), 256):\n            q = ix[start:start + 256]\n            sim = c4.X[q] @ c4.X[ix].T\n            sim[np.arange(len(q)), np.searchsorted(ix, q)] = -9.0\n            for row, nbs in zip(q, np.argsort(-sim, 1)[:, :k]):\n                wanted.update((min(int(row), int(j)), max(int(row), int(j)))\n                              for j in ix[nbs])\n    missing = sorted(wanted - cache.done.keys())\n    print(f"  SIFT: {len(wanted)} pairs needed, {len(missing)} uncached", flush=True)\n    if missing:\n        cache.genericness(train_idx, os.path.join(cache_dir, "keypoint_generic_3e.npz"))\n        t0 = time.time()\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            for s0 in range(0, len(missing), 20000):\n                block = missing[s0:s0 + 20000]\n                for pair, value in zip(block, pool.map(cache.verify, block)):\n                    cache.done[pair] = value\n                cache.save()\n                print(f"    verified {s0 + len(block)}/{len(missing)}  "\n                      f"({time.time() - t0:.0f}s)", flush=True)\n    pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n    vv = np.array([cache.done[tuple(p)] for p in pp], float).reshape(-1, 6)\n    a, b = pp.T\n\n    def mat(v):\n        return sparse.coo_matrix((np.r_[v, v], (np.r_[a, b], np.r_[b, a])),\n                                 shape=(n, n)).tocsr()\n\n    c4.GOOD, c4.INL = mat(vv[:, 0].astype(int)), mat(vv[:, 1].astype(int))\n    for j, name in enumerate(EXTRA_NAMES, 2):\n        EXTRA[name] = mat(vv[:, j].astype(np.float32))\n\n\n# ═══ PLDA and the extra features ═════════════════════════════════════════════\ndef plda_scores(gi, qi, inv, n, dim=128, shrink=0.1, n_cap=4):\n    """Two-covariance PLDA log likelihood ratio: same identity vs unrelated animal."""\n    key=(\'plda\',str(c4.SP[gi[0]]))\n    if c4.phase == \'load\':\n        projector,zcentre,centre,V,pm,pv,psi,d=c4.state[key]\n        uq=(c4.unit(projector.transform(c4.X[qi])-zcentre)*np.sqrt(d)-centre) @ V\n    else:\n        allx = c4.X[np.r_[gi, qi]]\n        d = min(dim, len(allx) - 1, allx.shape[1])\n        projector = PCA(d, svd_solver="randomized", random_state=0).fit(allx)\n        z = projector.transform(allx)\n        zcentre = z[:len(gi)].mean(0)\n        z = c4.unit(z - zcentre) * np.sqrt(d)\n        zg, zq = z[:len(gi)], z[len(gi):]\n        count = np.bincount(inv, minlength=n).astype(float)\n        means = np.zeros((n, d))\n        np.add.at(means, inv, zg)\n        means /= count[:, None]\n        res = zg - means[inv]\n        Sw = res.T @ res / max(len(gi) - n, 1)\n        Sw = (1 - shrink) * Sw + shrink * (np.trace(Sw) / d + 1e-6) * np.eye(d)\n        mu = means - means.mean(0)\n        psi, V = eigh(mu.T @ mu / max(n - 1, 1), Sw)          # V\' Sw V = I\n        psi = np.maximum(psi - np.mean(1 / count), 1e-4)       # noise in the class means\n        centre = zg.mean(0)\n        uq, ug = (zq - centre) @ V, (means - centre) @ V\n        ne = np.minimum(count, n_cap)[:, None]                 # images are not independent\n        pm = ne * psi / (ne * psi + 1) * ug\n        pv = 1 + psi / (ne * psi + 1)\n        if c4.phase == \'save\': c4.state[key]=(projector,zcentre,centre,V,pm,pv,psi,d)\n    same = -0.5 * ((uq ** 2) @ (1 / pv).T - 2 * uq @ (pm / pv).T\n                   + (pm ** 2 / pv + np.log(pv)).sum(1))\n    diff = -0.5 * ((uq ** 2) @ (1 / (1 + psi)) + np.log(1 + psi).sum())\n    return same - diff[:, None]\n\n\ndef species_features(gi, qi, sp):\n    q, cd = _base_species_features(gi, qi, sp)\n    ids, inv = np.unique(c4.Y[gi], return_inverse=True)\n    n = len(ids)\n    r, c = cd["q"].to_numpy(), cd["c"].to_numpy()\n\n    def add(name, M):\n        top = -np.sort(-M, 1)[:, :2]\n        cd[f"{name}_v"] = M[r, c]\n        cd[f"{name}_gap"] = M[r, c] - top[r, 0]\n        cd[f"{name}_rk"] = (-M).argsort(1).argsort(1)[r, c]\n        q[f"{name}_top"] = top[:, 0]\n        q[f"{name}_m12"] = top[:, 0] - top[:, 1]\n\n    for name in EXTRA_NAMES:\n        S = EXTRA[name][qi][:, gi].toarray().astype(np.float32)\n        best, top2 = c4.per_identity(S, inv, n)\n        add(name, best)\n        cd[f"{name}_top2"] = top2[r, c]\n    # Raw likelihood ratios change scale with gallery size (sim vs full train), so\n    # they are standardised per query and absolute top values are not used.\n    L = plda_scores(gi, qi, inv, n)\n    L = (L - L.mean(1, keepdims=True)) / np.maximum(L.std(1, keepdims=True), 1e-9)\n    add("plda", L)\n    del q["plda_top"]\n    return q, cd\n\n\nc4.species_features = species_features\n\n\n# ═══ simulation that follows the test viewpoint mix ══════════════════════════\ndef viewpoint_weights(train_idx, test_idx):\n    """Per image: test share / train share of its (species, viewpoint), clipped."""\n    w = np.ones(len(c4.X))\n    for sp in SPECIES:\n        tr = train_idx[c4.SP[train_idx] == sp]\n        te = test_idx[c4.SP[test_idx] == sp]\n        if not len(tr) or not len(te):\n            continue\n        for v in np.unique(c4.VIEW[tr]):\n            ratio = np.mean(c4.VIEW[te] == v) / np.mean(c4.VIEW[tr] == v)\n            w[tr[c4.VIEW[tr] == v]] = np.clip(ratio, 0.35, 3.0)\n    return w\n\n\ndef simulate(seed, train_idx, view_w=None):\n    """4c\'s open-set episode; held-out images are drawn in proportion to view_w."""\n    if view_w is None:\n        return c4.simulate(seed, train_idx)\n    rng = np.random.default_rng(seed)\n    link = c4.REGIMES[seed % len(c4.REGIMES)]\n    gallery, known, unknown = [], [], []\n    for sp in SPECIES:\n        p_unknown = rng.uniform(0.2, 0.35)\n        frac = rng.uniform(*c4.FRAC)\n        cap = int(rng.integers(12, 33))\n        ix = train_idx[c4.SP[train_idx] == sp]\n        names = np.array(sorted(set(c4.Y[ix])))\n        rng.shuffle(names)\n        new_set = set(names[:int(round(p_unknown * len(names)))])\n        for name in names:\n            rows = ix[c4.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in new_set:\n                keys = rng.random(len(rows)) ** (1 / view_w[rows])\n                unknown.extend(rows[np.argsort(-keys)][:cap].tolist())\n            elif len(rows) < 2:\n                gallery.extend(rows.tolist())\n            else:\n                n_q = min(max(int(round(frac * len(rows))), 1), cap, len(rows) - 1)\n                groups = c4.duplicate_groups(rows, link)\n                uniq = np.unique(groups)\n                gw = np.array([view_w[rows[groups == g]].mean() for g in uniq])\n                held = []\n                for g in uniq[np.argsort(-rng.random(len(uniq)) ** (1 / gw))]:\n                    members = rows[groups == g].tolist()\n                    if len(held) < n_q and len(held) + len(members) < len(rows):\n                        held.extend(members)\n                known.extend(held[:cap])\n                held_set = set(held)\n                gallery.extend(i for i in rows if i not in held_set)\n    gi = np.array(gallery, dtype=int)\n    qi = np.array(known + unknown, dtype=int)\n    frames = c4.build_frames(gi, qi)\n    truth = np.where(np.isin(frames["order"], unknown), NEW, c4.Y[frames["order"]])\n    frames["truth"] = truth\n    frames["q"]["unk"] = (truth == NEW).astype(int)\n    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) ==\n                        truth[frames["c"].q.to_numpy()]).astype(int)\n    return frames\n\n\ndef adversarial(sims, final):\n    """Can a classifier tell real test queries from simulated ones?  Returns the AUC,\n    the most telling features and a weight per simulated query."""\n    cols = [x for x in c4.columns(sims[0])[0] if x in final["q"].columns]\n    A = pd.concat([s["q"][cols] for s in sims], ignore_index=True).astype(float)\n    B = final["q"][cols].astype(float).reset_index(drop=True)\n    x = pd.concat([A, B], ignore_index=True)\n    y = np.r_[np.zeros(len(A)), np.ones(len(B))]\n    group = np.r_[np.concatenate([s["order"] for s in sims]), final["order"]]\n    p, imp = np.zeros(len(y)), np.zeros(len(cols))\n    for tr, te in GroupKFold(5).split(x, y, group):      # repeated images stay together\n        m = lgb.LGBMClassifier(n_estimators=150, num_leaves=7, learning_rate=0.05,\n                               min_child_samples=100, reg_lambda=5.0, verbose=-1)\n        m.fit(x.iloc[tr], y[tr])\n        p[te] = m.predict_proba(x.iloc[te])[:, 1]\n        imp += m.booster_.feature_importance("gain")\n    ps = np.clip(p[:len(A)], 1e-4, 1 - 1e-4)\n    w = np.clip(np.sqrt(ps / (1 - ps) * len(A) / len(B)), 0.25, 4.0)\n    top = [(cols[i], imp[i] / max(imp.sum(), 1e-9)) for i in np.argsort(-imp)[:8]]\n    return roc_auc_score(y, p), top, w / w.mean()\n\n\n# ═══ identity-disjoint training ══════════════════════════════════════════════\ndef identity_folds(train_idx, n_folds, seed):\n    """One fold per animal, balanced by image count within each species."""\n    rng = np.random.default_rng(seed)\n    fold = {}\n    for sp in SPECIES:\n        names, counts = np.unique(c4.Y[train_idx[c4.SP[train_idx] == sp]],\n                                  return_counts=True)\n        load = np.zeros(n_folds)\n        for i in np.lexsort((rng.random(len(names)), -counts)):\n            f = int(load.argmin())\n            fold[names[i]] = f\n            load[f] += counts[i]\n    return fold\n\n\ndef fit_models(Q, C, unk, y, qmask, w):\n    """4c\'s three bagged classifiers, trained on the queries selected by qmask."""\n    K = c4.K\n    cmask = np.repeat(qmask, K)\n    kmask = np.repeat(qmask & (unk == 0), K)\n    wc = np.repeat(w, K)\n\n    def bag(params, x, target, weight):\n        base = params.get("random_state", 0) or 0\n        return c4.Bag([lgb.LGBMClassifier(**{**params, "random_state": base + 101 * k})\n                       .fit(x, target, sample_weight=weight) for k in range(c4.SEEDS)])\n\n    det = {**c4.PARAMS, "n_estimators": 300,\n           "num_leaves": min(15, c4.PARAMS["num_leaves"])}\n    return (bag(det, Q[qmask], unk[qmask], w[qmask]),\n            bag(c4.PARAMS, C[kmask], y[kmask], wc[kmask]),\n            bag(c4.PARAMS, C[cmask], y[cmask], wc[cmask]))\n\n\ndef stack(sims, qfold, weights, label, rounds=None, keep_models=True):\n    """Multi-round stacking with animal-disjoint folds.  weights=None means uniform."""\n    rounds = rounds or c4.ROUNDS\n    sizes = [len(s["q"]) for s in sims]\n    cuts = np.cumsum(sizes)[:-1]\n    fold = np.concatenate(qfold)\n    w = np.ones(len(fold)) if weights is None else weights\n    round_models = []\n    for rnd in range(1, rounds + 1):\n        qf, cf = c4.columns(sims[0])\n        Q = pd.concat([s["q"][qf] for s in sims], ignore_index=True).astype(np.float32)\n        C = pd.concat([c4.candidate_matrix(s, qf, cf) for s in sims],\n                      ignore_index=True).astype(np.float32)\n        unk = np.concatenate([s["q"].unk.to_numpy() for s in sims])\n        y = np.concatenate([s["c"].y.to_numpy() for s in sims])\n        res = {k: np.zeros((len(fold), c4.K)) for k in ("raw", "pc")}\n        res.update(pn=np.zeros(len(fold)), pb=np.zeros(len(fold)))\n        for f in np.unique(fold):\n            test = fold == f\n            ctest = np.repeat(test, c4.K)\n            det, closed, anyq = fit_models(Q, C, unk, y, ~test, w)\n            raw = anyq.predict_proba(C[ctest])[:, 1].reshape(-1, c4.K)\n            within = closed.predict_proba(C[ctest])[:, 1].reshape(-1, c4.K)\n            res["raw"][test] = raw\n            res["pc"][test] = within / np.maximum(within.sum(1, keepdims=True), 1e-9)\n            res["pn"][test] = det.predict_proba(Q[test])[:, 1]\n            res["pb"][test] = (res["pn"][test] + np.clip(1 - raw.sum(1), 0, 1)) / 2\n        if keep_models:\n            round_models.append((*fit_models(Q, C, unk, y, np.ones(len(fold), bool), w),\n                                 (qf, cf)))\n        oof = [{k: v for k, v in zip(res, vals)} for vals in\n               zip(*[np.split(res[k], cuts) for k in res])]\n        ap = np.concatenate([ap_of(s, fusion(s, r)) for s, r in zip(sims, oof)])\n        print(f"  [{label}] round {rnd}: MAP@5 = {ap.mean():.4f}", flush=True)\n        if rnd < rounds:\n            sims = [c4.augment(s, r) for s, r in zip(sims, oof)]\n    return dict(models=round_models, sims=sims, oof=oof, ap=ap)\n\n\n# ═══ scores, the single ranker and the comparison ════════════════════════════\ndef fusion(s, r):\n    """4c\'s scores over K candidates + new_individual (rows sum to 1)."""\n    return np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n\n\ndef top5(s, scores):\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, c4.K),\n                             np.full((nq, 1), NEW, dtype=object)], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1, kind="stable")[:, :5], 1)\n\n\ndef ap_of(s, scores):\n    hit = top5(s, scores) == s["truth"][:, None]\n    return (hit / np.arange(1, 6)).sum(1)\n\n\ndef rank_matrix(s, qf, cf):\n    """(queries x (K+1), features): K candidate rows then one new_individual row."""\n    nq, K = len(s["q"]), c4.K\n    C = s["c"][cf].to_numpy(np.float32).reshape(nq, K, len(cf))\n    C = np.concatenate([C, np.full((nq, 1, len(cf)), np.nan, np.float32)], 1)\n    Q = np.repeat(s["q"][qf].to_numpy(np.float32)[:, None, :], K + 1, 1)\n    flag = np.zeros((nq, K + 1, 1), np.float32)\n    flag[:, K] = 1\n    return np.concatenate([C, Q, flag], 2).reshape(nq * (K + 1), -1)\n\n\ndef fit_ranker(X, rel, qmask, w):\n    """rel: (queries, K+1) relevance.  Queries with no correct row are skipped."""\n    use = qmask & (rel.sum(1) > 0)\n    rows = np.repeat(use, rel.shape[1])\n    params = {k: v for k, v in c4.PARAMS.items() if k != "objective"}\n    return [lgb.LGBMRanker(objective="lambdarank", lambdarank_truncation_level=8,\n                           **{**params, "random_state": 11 + 101 * k})\n            .fit(X[rows], rel[use].ravel(), group=np.full(use.sum(), rel.shape[1]),\n                 sample_weight=np.repeat(w[use], rel.shape[1]))\n            for k in range(c4.SEEDS)]\n\n\ndef ranker_scores(models, X):\n    z = np.mean([m.predict(X, raw_score=True) for m in models], 0).reshape(-1, c4.K + 1)\n    e = np.exp(z - z.max(1, keepdims=True))\n    return e / e.sum(1, keepdims=True)\n\n\ndef final_layer(run, qfold, weights):\n    """Out-of-fold ranker on top of the last round; returns scores for three choices."""\n    sims = [c4.augment(s, r) for s, r in zip(run["sims"], run["oof"])]\n    qf, cf = c4.columns(sims[0])\n    X = np.concatenate([rank_matrix(s, qf, cf) for s in sims])\n    rel = np.concatenate([np.concatenate(\n        [s["c"].y.to_numpy().reshape(-1, c4.K), s["q"].unk.to_numpy()[:, None]], 1)\n        for s in sims])\n    fold = np.concatenate(qfold)\n    w = np.ones(len(fold)) if weights is None else weights\n    scores = np.zeros((len(fold), c4.K + 1))\n    for f in np.unique(fold):\n        test = fold == f\n        scores[test] = ranker_scores(fit_ranker(X, rel, ~test, w),\n                                     X[np.repeat(test, c4.K + 1)])\n    parts = np.split(scores, np.cumsum([len(s["q"]) for s in sims])[:-1])\n    options = {"fusion": [fusion(s, r) for s, r in zip(sims, run["oof"])],\n               "ranker": parts}\n    options["blend"] = [(a + b) / 2 for a, b in zip(options["fusion"], parts)]\n    ap = {k: np.concatenate([ap_of(s, m) for s, m in zip(sims, v)])\n          for k, v in options.items()}\n    full = (fit_ranker(X, rel, np.ones(len(fold), bool), w), qf, cf)\n    return ap, full, options\n\n\ndef bootstrap(diff, animals, weights=None, reps=2000, seed=0):\n    """Mean difference and 95% interval, resampling whole animals."""\n    w = np.ones(len(diff)) if weights is None else weights\n    _, g = np.unique(animals, return_inverse=True)\n    num, den = np.bincount(g, diff * w), np.bincount(g, w)\n    rng = np.random.default_rng(seed)\n    draws = rng.integers(0, len(num), (reps, len(num)))\n    boot = num[draws].sum(1) / den[draws].sum(1)\n    return num.sum() / den.sum(), np.quantile(boot, [0.025, 0.975])\n\n\ndef prior_em(p, train_prior, iters=200):\n    """Saerens et al. 2002: re-estimate the new-animal share from unlabelled scores."""\n    pi = train_prior\n    for _ in range(iters):\n        a = p * pi / train_prior\n        post = a / (a + (1 - p) * (1 - pi) / (1 - train_prior))\n        pi = float(np.clip(post.mean(), 1e-3, 1 - 1e-3))\n    return pi, post\n\n\nRETUNE_GRID = [(31, 40, 2.0), (15, 40, 2.0), (15, 100, 5.0),\n               (31, 100, 5.0), (7, 100, 10.0), (63, 40, 2.0)]\n\n\n'
exec(compile(_MODEL3E_SOURCE, "<embedded Model 3e>", "exec"), globals())

# Match 3e defaults.
c4.K = K_CANDIDATES
c4.ROUNDS = N_ROUNDS
c4.NEIGHBOURS = N_NEIGHBOURS
c4.SEEDS = N_SEEDS
c4.PARAMS.update(n_estimators=N_TREES, random_state=RANDOM_SEED)
print("Embedded Model 3e definitions loaded; no Dewang-module import used.")
c4.phase='episode'
c4.state={}
''')
exec(runtime_parts[-1])


## Paired evaluation
Reserve one of five balanced animal groups before fitting the evaluation models. Six new episodes on these animals measure the baseline and extension on identical queries. The full-gallery production control uses 24 episodes, five folds and three seeds. Its final layer averages the 3e fusion and LambdaRank probabilities.

In [ ]:
runtime_parts.append(r'''if RAW is None:
    train_df, test_df, master, train_idx, test_idx = c4.load_data(str(ROOT))
    sample_path = ROOT / "DATA/reid_student_data_main/sample_submission.csv"
else:
    train_df = pd.read_csv(RAW/"train.csv")
    test_df = pd.read_csv(RAW/("test_public.csv" if (RAW/"test_public.csv").exists() else "test.csv"))
    master = pd.concat([train_df,test_df],ignore_index=True)
    feature_ids = pd.Index(pd.read_csv(RAW/"miewid_ids.csv").id)
    alignment = feature_ids.get_indexer(master.id)
    assert master.id.is_unique and feature_ids.is_unique and np.all(alignment >= 0)
    c4.X = c4.unit(np.load(RAW/"miewid_features.npy",mmap_mode="r")[alignment])
    c4.IDS = master.id.to_numpy()
    c4.Y = master.identity.to_numpy(object)
    c4.SP = master.species.to_numpy()
    c4.VIEW = master.viewpoint.map(dict(zip(c4.VIEWS,range(len(c4.VIEWS))))).fillna(-1).to_numpy(int)
    train_idx = np.arange(len(train_df))
    test_idx = np.arange(len(train_df),len(master))
    sample_path = RAW/"sample_submission.csv"

    class SplitSift:
        def __init__(self):
            self.locations = {}
            for part in ("train","test"):
                matches = sorted(Path("/kaggle/input").rglob(f"sift_features_{part}.npz"))
                if not matches:
                    raise FileNotFoundError(f"Attach the supplied {part} SIFT descriptors")
                source = matches[0]
                if source.is_dir():
                    self.locations.update({p.stem:(p,None) for p in source.rglob("*.npy")})
                else:
                    archive = np.load(source,allow_pickle=False)
                    self.locations.update({Path(k).name:(archive,k) for k in archive.files})
            self.files = list(self.locations)
        def __getitem__(self,key):
            obj,k = self.locations[key]
            return np.load(obj,allow_pickle=False) if k is None else obj[k]

    c4.raw_sift = SplitSift()
    assert all(f"{i}_desc" in c4.raw_sift.locations and f"{i}_xy" in c4.raw_sift.locations for i in c4.IDS)

original_frames = c4.build_frames
def frames_with_gallery(gi, qi):
    return dict(original_frames(gi, qi), gallery=gi.copy())
c4.build_frames = frames_with_gallery
''')
exec(runtime_parts[-1])
sift_matrices(str(PROC / 'sift_features.npz'), str(CACHE), train_idx, N_WORKERS)
view_w = viewpoint_weights(train_idx, test_idx)
partition = identity_folds(train_idx, 5, 7719)
audit = np.array([i for i in train_idx if partition[c4.Y[i]] == 0])
dev = np.setdiff1d(train_idx, audit)
assert set(c4.Y[dev]).isdisjoint(c4.Y[audit])


In [ ]:
runtime_parts.append(r'''def fit_control(pool, count, seeds):
    c4.SEEDS = seeds
    sims = [simulate(42+i, pool, view_w) for i in range(count)]
    fold_of = identity_folds(pool, 5, 42)
    folds = [np.array([fold_of[c4.Y[i]] for i in s['order']]) for s in sims]
    run = stack(sims, folds, None, '3e', keep_models=True)
    _, ranker, options = final_layer(run, folds, None)
    return run['models'], ranker, [packet(s, m) for s,m in zip(sims,options['blend'])]

def packet(s, scores):
    return dict(query=s['order'], gallery=s['gallery'], scores=scores,
                labels=s['c'].ident.to_numpy(str).reshape(-1,c4.K),
                truth=np.asarray(s.get('truth', []),str))

def score_control(models, ranker, s):
    for model in models:
        result = c4.predict(model, s)
        s = c4.augment(s, result)
    rm,qf,cf = ranker
    score = (fusion(s,result) + ranker_scores(rm,rank_matrix(s,qf,cf)))/2
    return packet(s,score)

# Keep the probability packets for error analysis and reproducible paired comparisons.
def save_packets(name, packets):
    for i,p in enumerate(packets):
        np.savez_compressed(CACHE / f'm8_{name}_{i}.npz', **p)

''')
exec(runtime_parts[-1])
with threadpool_limits(limits=4, user_api='blas'):
    models, ranker, development = fit_control(dev, 10, 1)
    save_packets('development', development)
    audit_rng = np.random.get_state()
    held = [score_control(models, ranker, simulate(8550+i,audit,view_w)) for i in range(6)]
    save_packets('audit', held)
    # The same evaluation animals face all development animals as gallery distractors.
    production_rng = np.random.get_state()
    np.random.set_state(audit_rng)
    large = []
    for j in range(3):
        small = simulate(91531+j, audit, view_w)
        lookup = dict(zip(small['order'],small['truth']))
        expanded = c4.build_frames(np.r_[small['gallery'],dev],small['order'])
        expanded['truth'] = np.array([lookup[i] for i in expanded['order']])
        large.append(score_control(models,ranker,expanded))
    np.random.set_state(production_rng)
    del models, ranker
    models, ranker, _ = fit_control(train_idx, 24, 3)
    c4.phase="save"
    competition = score_control(models, ranker, c4.build_frames(train_idx,test_idx))
    c4.phase='episode'
    save_packets('test',[competition])


## Texture retrieval using VLAD
Learn 64 visual words on RootSIFT descriptors from development animals. Each sighting pools its descriptor residuals by visual word. This representation adds candidates based on local coat texture even when geometric matching yields few inliers.

In [ ]:
runtime_parts.append(r'''from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
def texture_vectors(training_rows):
    """VLAD over RootSIFT, with a vocabulary learned on development animals."""
    rng=np.random.default_rng(806)
    store=c4.SiftCache(str(PROC/'sift_features.npz'),str(CACHE/'unused_texture_cache.npz'),1)
    read=lambda i: store._descriptors_raw(int(i))[0]
    # Limit each animal's influence on the texture vocabulary.
    if c4.phase=='load':
        projector,vocab,pca=c4.state['texture']
    else:
        samples=[]
        for animal in np.unique(c4.Y[training_rows]):
            group=training_rows[c4.Y[training_rows]==animal]
            for i in rng.choice(group,min(5,len(group)),replace=False):
                d=read(i)
                if len(d): samples.append(d[rng.choice(len(d),min(45,len(d)),replace=False)])
        samples=np.concatenate(samples)
        projector=PCA(48,svd_solver='randomized',random_state=806).fit(samples)
        vocab=MiniBatchKMeans(64,batch_size=2048,n_init=2,random_state=806).fit(projector.transform(samples))
    centers=vocab.cluster_centers_
    vec=np.zeros((len(c4.X),centers.size),np.float32)
    for i in range(len(vec)):
        d=read(i)
        if len(d):
            z=projector.transform(d)
            word=vocab.predict(z)
            accum=np.zeros_like(centers)
            np.add.at(accum,word,z-centers[word])
            v=c4.unit(accum).ravel()
            vec[i]=np.sign(v)*np.sqrt(np.abs(v))
    vec=c4.unit(vec)
    if c4.phase!='load':
        pca=PCA(192,svd_solver='randomized',random_state=806).fit(vec[training_rows])
        c4.state['texture']=(projector,vocab,pca)
    return c4.unit(pca.transform(vec)/(pca.explained_variance_+1e-6)**0.25)
''')
exec(runtime_parts[-1])
with threadpool_limits(limits=3):
    texture=texture_vectors(dev)


## Same-animal link classifier
Each species has its own gallery-trained verifier. It uses raw and partially whitened embedding similarities, local neighbor density, mutual neighbors and four SIFT geometry statistics. Strong links pool the 3e probabilities across one and two neighboring sightings. Direct gallery matches also propose additional candidates.

In [ ]:
runtime_parts.append(r'''def neighbors(sim, k):
    k = min(k, sim.shape[1])
    return np.argpartition(-sim, k-1, axis=1)[:, :k]

def pair_features(a, b, raw, white, same_pool):
    """Retrieve candidate pairs independently in both embedding spaces and by geometry."""
    rs, ws = c4.X[a] @ c4.X[b].T, raw @ white.T
    ts=texture[a] @ texture[b].T
    if same_pool:
        np.fill_diagonal(ts,-2)
        np.fill_diagonal(rs,-2)
        np.fill_diagonal(ws,-2)
    ks = min(12, len(b)-1, len(a)-1)
    rr, wr = neighbors(rs,ks), neighbors(ws,ks)
    tr=neighbors(ts,ks)
    td=np.take_along_axis(ts,tr,1).mean(1)
    reverse = neighbors(rs.T,ks)
    row_density = np.take_along_axis(rs,rr,axis=1).mean(1)
    col_density = np.take_along_axis(rs.T,reverse,axis=1).mean(1)
    inl = c4.INL[a][:, b].toarray()
    ir = neighbors(inl,ks)
    row = np.broadcast_to(np.arange(len(a))[:,None],rr.shape)
    has_geometry = inl[row,ir] >= 4
    codes=np.unique(np.concatenate([(row*len(b)+rr).ravel(), (row*len(b)+wr).ravel(), (row*len(b)+tr).ravel(),
                                    (row*len(b)+ir)[has_geometry]]))
    i,j=np.divmod(codes,len(b))
    if same_pool:
        keep=i!=j
        i,j=i[keep],j[keep]
    ai,bj=a[i],b[j]
    good=np.asarray(c4.GOOD[ai,bj]).ravel()
    f={'texture':ts[i,j], 'texture_margin':ts[i,j]-td[i], 'cos':rs[i,j], 'whitened':ws[i,j], 'excess_src':rs[i,j]-row_density[i],
       'excess_dst':rs[i,j]-col_density[j], 'src_rank12':(rr[i]==j[:,None]).any(1),
       'reciprocal':(reverse[j]==i[:,None]).any(1), 'inliers':inl[i,j],
       'inlier_fraction':inl[i,j]/np.maximum(good,1), 'matches':good,
       'view_match':(c4.VIEW[ai]==c4.VIEW[bj]) & (c4.VIEW[ai]>=0)}
    for name in EXTRA_NAMES:
        f[name]=np.asarray(EXTRA[name][ai,bj]).ravel()
    return i,j,pd.DataFrame(f)

def pair_graph(g,q):
    # Fit both representation and pair classifier afresh for each episode / species.
    from sklearn.decomposition import PCA
    key=('pair',str(c4.SP[g[0]]))
    if c4.phase=='load':
        pca,zg,classifier=c4.state[key]
        zq=c4.unit(pca.transform(c4.X[q])*(pca.explained_variance_+1e-6)**(-0.35))
    else:
        pca=PCA(n_components=min(128,len(g)-1),svd_solver='randomized',random_state=31)
        z=pca.fit_transform(c4.X[g])
        scaling=(pca.explained_variance_+1e-6)**(-0.35)
        zg,zq=c4.unit(z*scaling),c4.unit(pca.transform(c4.X[q])*scaling)
        a,b,f=pair_features(g,g,zg,zg,True)
        classifier=lgb.LGBMClassifier(n_estimators=120,num_leaves=15,min_child_samples=60,
                                      reg_lambda=5,learning_rate=.07,n_jobs=3,verbosity=-1,random_state=417)
        classifier.fit(f,(c4.Y[g[a]]==c4.Y[g[b]]).astype(int))
        if c4.phase=='save': c4.state[key]=(pca,zg,classifier)
    def links(target,zt,self_links):
        a,b,x=pair_features(q,target,zq,zt,self_links)
        p=classifier.predict_proba(x)[:,1].astype(np.float32)
        return sparse.csr_matrix((p,(a,b)),shape=(len(q),len(target)))
    return links(q,zq,True),links(g,zg,False)

def gallery_kernel(g, q, inv):
    # Solve a regularized least-squares identification problem for all gallery identities.
    # The cubic term adds local matches; the linear term combines evidence across views.
    from scipy.linalg import cho_factor, cho_solve
    def gram(ix):
        visual = c4.X[ix] @ c4.X[g].T
        tex = texture[ix] @ texture[g].T
        return .55*visual + .30*visual**3 + .15*tex
    key=('kernel',str(c4.SP[g[0]]))
    if c4.phase=='load': weights=c4.state[key]
    else:
        kernel=gram(g)
        counts=np.bincount(inv)
        kernel.flat[::len(g)+1] += .20 * np.sqrt(counts[inv])
        y=np.eye(len(counts),dtype=np.float32)[inv]
        weights=cho_solve(cho_factor(kernel,lower=True,check_finite=False),y,check_finite=False)
        if c4.phase=='save': c4.state[key]=weights
    return gram(q) @ weights

def candidate_frame(packet):
    frames=[]
    for sp in SPECIES:
        rows=np.where(c4.SP[packet['query']]==sp)[0]
        q=packet['query'][rows]
        g=packet['gallery'][c4.SP[packet['gallery']]==sp]
        names,inv=np.unique(c4.Y[g].astype(str),return_inverse=True)
        n=len(names)
        ids=np.searchsorted(names,packet['labels'][rows])
        prior=np.zeros((len(q),n+1),np.float32)
        rid=np.arange(len(q))[:,None]
        prior[rid,ids]=packet['scores'][rows,:-1]
        ridge=np.zeros_like(prior)
        ridge[:,:n]=gallery_kernel(g,q,inv)
        prior[:,-1]=packet['scores'][rows,-1]
        graph,anchors=pair_graph(g,q)
        # Make the links symmetric. Discount weak matches before pooling.
        graph=(graph+graph.T)*.5
        graph.data **= 3
        mass=np.asarray(graph.sum(1)).ravel()
        norm=sparse.diags(1/np.maximum(mass,1e-6)) @ graph
        pooled=norm @ prior
        second=norm @ pooled
        strong=np.asarray(graph.max(1).toarray()).ravel()
        a=anchors.tocoo()
        direct=np.zeros_like(prior)
        np.maximum.at(direct,(a.row,inv[a.col]),a.data)
        anchor_mass=np.zeros_like(prior)
        np.add.at(anchor_mass,(a.row,inv[a.col]),a.data**3)
        # Candidates include direct gallery matches and identities proposed by adjacent queries.
        shortlist=np.zeros_like(prior,dtype=bool)
        shortlist[rid,ids]=True
        for scores in (direct,pooled,second,ridge):
            best=neighbors(scores[:,:n],6)
            shortlist[rid,best] |= scores[rid,best]>0
        shortlist[:,-1]=True
        i,j=np.where(shortlist)
        def margin(s): return s[i,j]-s.max(1)[i]
        f=pd.DataFrame(dict(row=rows[i],name=np.append(names,NEW)[j],score=prior[i,j],
            base_gap=margin(prior),base_new=prior[i,-1],base_peak=prior.max(1)[i],
            near=pooled[i,j],near_gap=margin(pooled),two_step=second[i,j],
            neighbor_new=pooled[i,-1],neighbor_strength=strong[i],neighbor_mass=mass[i],
            gallery_match=direct[i,j],gallery_gap=margin(direct),gallery_mass=np.log1p(anchor_mass[i,j]),
            kernel_score=ridge[i,j],kernel_gap=margin(ridge),kernel_peak=ridge.max(1)[i],
            gallery_peak=direct.max(1)[i],is_new=j==n,is_hyena=sp=='hyena'))
        if len(packet['truth']):
            f['correct']=f.name.to_numpy()==packet['truth'][rows[i]]
        frames.append(f)
    return pd.concat(frames,ignore_index=True).sort_values('row',kind='stable').reset_index(drop=True)
''')
exec(runtime_parts[-1])


## Ranking and competition export
Regularized kernel regression uses the sighting's embedding and texture to score every labelled animal in the gallery. A classifier learns to combine those scores with direct and neighboring evidence on development animals. Four animal folds within development choose how strongly to blend this classifier with 3e. Six episodes on separately reserved animals measure the selected model. Intervals resample whole animals across episodes.

In [ ]:
runtime_parts.append(r'''from pathlib import Path
from scipy import sparse
def fit_extension(frames):
    df=pd.concat(frames,ignore_index=True)
    cols=[k for k in df if k not in ('row','name','correct')]
    model=lgb.LGBMClassifier(n_estimators=240,num_leaves=12,min_child_samples=100,
                             reg_lambda=8,learning_rate=.045,random_state=884,n_jobs=4,verbosity=-1)
    return model.fit(df[cols],df.correct.astype(int)),cols

def ranked(f, extension, strength):
    model,cols=extension
    x=f.copy()
    prediction=model.predict_proba(x[cols])[:,1]
    totals=np.bincount(x.row,weights=prediction)
    x['combined']=(1-strength)*x.score + strength*prediction/np.maximum(totals[x.row],1e-9)
    sort=x.sort_values(['row','combined'],ascending=[True,False],kind='stable')
    return sort.groupby('row',sort=True).head(5).name.to_numpy().reshape(-1,5)

def ap(truth,pred):
    return ((pred==truth[:,None])/np.arange(1,6)).sum(1)

def choose_strength(frames, packets):
    # Each tuning animal receives predictions from a meta-model trained on other animals.
    # Repeated sightings of that animal stay in the same tuning fold across episodes.
    folds=identity_folds(dev, 4, 420)
    groups=[np.array([folds[c4.Y[i]] for i in p['query']]) for p in packets]
    choices=(.35,.65,1.0)
    totals=np.zeros(len(choices))
    baseline=0.0
    count=0
    for fold in range(4):
        fitframes=[f.loc[grp[f.row.to_numpy()]!=fold] for f,grp in zip(frames,groups)]
        model=fit_extension(fitframes)
        for f,p,grp in zip(frames,packets,groups):
            valid=grp==fold
            base=ap(p['truth'],ranked(f,model,0))
            baseline+=base[valid].sum()
            count+=valid.sum()
            for j,t in enumerate(choices):
                totals[j]+=ap(p['truth'],ranked(f,model,t))[valid].sum()
    print('Development MAP@5, matched baseline',baseline/count,'blend strengths',dict(zip(choices,totals/count)))
    return choices[int(totals.argmax())]


def input_signatures():
    # Include descriptor values as well as row IDs to prevent stale geometry-cache reuse.
    reader=c4.SiftCache(str(PROC/'sift_features.npz'),str(CACHE/'unused_signature_cache.npz'),1)
    sums=[]
    for ix in (train_idx,test_idx):
        h=hashlib.sha256()
        for i in ix:
            for value in (c4.IDS[i],c4.SP[i],c4.VIEW[i],c4.Y[i] if i in gallery_set else ''):
                h.update((str(value)+'\0').encode())
            h.update(c4.X[i].tobytes())
            d,xy=reader._descriptors_raw(i)
            h.update(d.tobytes());h.update(xy.tobytes())
        sums.append(h.hexdigest())
    return sums
gallery_set=set(train_idx.tolist())

def score_artifact(bundle):
    global texture
    payload=joblib.load(bundle/'model8.joblib')
    c4.state=payload['transforms']
    c4.phase='load'
    signatures=input_signatures()
    if signatures[0]!=payload['inputs'][0]: raise ValueError('Gallery input differs from the fitted model')
    if signatures==payload['inputs']:
        with np.load(bundle/'public_sift_pairs.npz') as cache:
            i,j=cache['pairs'].T
            vv=cache['values']
        def matrix(v):
            return sparse.coo_matrix((np.r_[v,v],(np.r_[i,j],np.r_[j,i])),shape=(len(c4.X),len(c4.X))).tocsr()
        c4.GOOD,c4.INL=matrix(vv[:,0].astype(int)),matrix(vv[:,1].astype(int))
        for k,key in enumerate(EXTRA_NAMES,2): EXTRA[key]=matrix(vv[:,k].astype(np.float32))
    else:
        fresh=CACHE/('geometry_'+signatures[0][:16]+signatures[1][:16])
        fresh.mkdir(exist_ok=True)
        sift_matrices(str(PROC/'sift_features.npz'),str(fresh),train_idx,N_WORKERS)
    models=[(c4.Bag(d),c4.Bag(cl),c4.Bag(anyq),cols) for d,cl,anyq,cols in payload['rounds']]
    with threadpool_limits(limits=4,user_api='blas'):
        test_packet=score_control(models,payload['ranker'],c4.build_frames(train_idx,test_idx))
    with threadpool_limits(limits=3):
        texture=texture_vectors(None)
    with threadpool_limits(limits=3,user_api='blas'):
        frame=candidate_frame(test_packet)
    pred=ranked(frame,payload['extension'],payload['strength'])
    result=pd.DataFrame({'id':c4.IDS[test_packet['query']], 'predictions':[' '.join(row) for row in pred]})
    result=pd.read_csv(sample_path)[['id']].merge(result,on='id',how='left',validate='one_to_one')
    c4.validate_submission(result,test_df,train_df,str(sample_path))
    return result
''')
exec(runtime_parts[-1])
# A separate set of animals measures improvement after fitting. Fix the blend before this evaluation.

with threadpool_limits(limits=3,user_api='blas'):
    dev_frames=[candidate_frame(p) for p in development]
    strength=choose_strength(dev_frames,development)
    extension=fit_extension(dev_frames)
    audit_frames=[candidate_frame(p) for p in held]
diffs=[]
records=[]
for p,f in zip(held,audit_frames):
    base=ap(p['truth'],ranked(f,extension,0))
    candidate=ap(p['truth'],ranked(f,extension,strength))
    diffs.append(candidate-base)
    records.append(pd.DataFrame(dict(animal=c4.Y[p['query']],species=c4.SP[p['query']],
                                     known=p['truth']!=NEW,control=base,candidate=candidate)))
report=pd.concat(records,ignore_index=True)
delta,ci=bootstrap(report.candidate-report.control,report.animal.to_numpy())
print(report[['control','candidate']].mean(), 'paired delta',delta,'95% interval',ci)
report.to_csv(CACHE/'m8_final_validation.csv',index=False)
print(report.groupby(['species','known'])[['control','candidate']].mean())
with threadpool_limits(limits=3,user_api='blas'):
    stress_records=[]
    for pkt in large:
        frame=candidate_frame(pkt)
        stress_records.append(pd.DataFrame(dict(animal=c4.Y[pkt['query']],species=c4.SP[pkt['query']],
            known=pkt['truth']!=NEW,control=ap(pkt['truth'],ranked(frame,extension,0)),
            candidate=ap(pkt['truth'],ranked(frame,extension,strength)))))
    stress=pd.concat(stress_records,ignore_index=True)
    stress.to_csv(CACHE/'m8_largegallery_validation.csv',index=False)
    print('Large gallery',stress[['control','candidate']].mean(),
          bootstrap(stress.candidate-stress.control,stress.animal.to_numpy()))
    c4.phase="save"
    test_features=candidate_frame(competition)
pred=ranked(test_features,extension,strength)
sample=pd.read_csv(sample_path)
submission=sample[['id']].merge(pd.DataFrame({'id':c4.IDS[competition['query']],
     'predictions':[' '.join(row) for row in pred]}),validate='one_to_one',how='left')
c4.validate_submission(submission,test_df,train_df,str(sample_path))
out=WORK/'submission_model8.csv'
submission.to_csv(out,index=False)
print(out,len(submission),'rows')


In [ ]:
# Fit outputs are stored as built-in containers plus fitted sklearn/LightGBM objects.
bundle=WORK/'model8_artifacts'
bundle.mkdir(exist_ok=True)
rounds=[(det.models,closed.models,anyq.models,cols) for det,closed,anyq,cols in models]
payload=dict(rounds=rounds,ranker=ranker,extension=extension,strength=strength,transforms=c4.state,inputs=input_signatures())
for sp in SPECIES:
    assert all((k,sp) in c4.state for k in ('embedding','plda','pair','kernel'))
assert 'texture' in c4.state
(bundle/'model8_runtime.py').write_text('\n'.join(runtime_parts))
shutil.copyfile(CACHE/'sift_pairs_3e.npz',bundle/'public_sift_pairs.npz')
joblib.dump(payload,bundle/'model8.tmp',compress=3)
(bundle/'model8.tmp').replace(bundle/'model8.joblib')
print('Exported',bundle)


## Measured local results · 27 November 2026

| Evaluation | Query appearances | 3e control | Model 8 | Difference | 95% animal bootstrap interval for difference |
|---|---:|---:|---:|---:|---:|
| Six episodes on reserved animals | 2,836 | 0.896186 | 0.899465 | +0.003279 | +0.000573 to +0.006186 |
| Three episodes with 6,233–6,667 gallery images | 1,386 | 0.862734 | 0.869252 | +0.006518 | +0.001325 to +0.012859 |

Each row uses unweighted query MAP@5. The 3e control for each evaluation uses ten training episodes and one boosting seed, with the same underlying 3e scores supplied to Model 8. Production uses the full 24-episode, three-seed 3e fit. The large-gallery check uses the same reserved animals, fresh episode seeds and additional gallery distractors. The two rows are related measurements.

Four development folds selected a 65% weight for the new scorer. Across the first six evaluation episodes, the model gained 32 first-place answers and lost 13. It placed four correct known animals in the top five that had been outside the original 15 candidates. Known-query MAP@5 improved from 0.828444 to 0.834942. Unknown-query MAP@5 moved from 0.982014 to 0.981215. Kaggle determines the public score from the exported CSV.


In [ ]:
# Single-file deployment for Kaggle. Load with Python's pickle.load.
import pickle
source=(bundle/'model8_runtime.py').read_text()
signature="def score_artifact(bundle):"
read_weights="    payload=joblib.load(bundle/'model8.joblib')"
assert signature in source and read_weights in source
source=source.replace(signature,"def score_artifact(bundle, payload):").replace(read_weights,"")
package=dict(format='model8-v1',model=payload,feature_code=source,
             public_geometry=(bundle/'public_sift_pairs.npz').read_bytes())
path=WORK/'model.pkl'
with path.with_suffix('.pkl.tmp').open('wb') as stream:
    pickle.dump(package,stream,protocol=5)
path.with_suffix('.pkl.tmp').replace(path)
print('Fitted model exported to',path)
